# Heart Vignette

## Purpose

Determine how different heart cell types arise from the mesoderm and heart fields.

## Setup

In [1]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import pycea as py
import scanpy as sc
import seaborn as sns
from devmap.config import (
    colors,
    cooc_cmap,
    germ_layer_palette,
    get_paths,
    sequential_cmap,
    set_theme,
    stage_palette,
    subtype_palette,
)
from devmap.fate import annotate_commitment, extant_transition_count, get_commitment_path
from devmap.plots import plot_progenitor_output, plot_restriction_sankey
from devmap.progenitors import compute_pmi, get_fate_progenitors, progenitor_dispersion_stratified_permutation
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("vignettes")

## Load data

In [2]:
tdata = load_data("log1p_hvg", scvi = True)
heart_umap = pd.read_csv(results_path / "heart_umap.csv", index_col=0)

## Set heart field

In [4]:
heart_fields = {'Inflow tract': 'pSHF',
                'Atria': 'pSHF',
                'Second heart field (posterior)': 'pSHF',
                'Right ventricle': 'aSHF',
                'Outflow tract': 'aSHF',
                'Second heart field (anterior)': 'aSHF',
                'Juxtacardiac field': 'JCF',
                'Epicardium': 'Epicardium',
                'Endocardial cell': 'Endocardial',
                'Left ventricle': 'FHF',
                'Atrioventricular canal': 'FHF',
                'First heart field': 'FHF'}
field_order = ["pSHF", "aSHF","JCF", "Epicardium","Endocardial","FHF"]
tdata.obs["heart_field"] = tdata.obs["cell_subtype"].map(heart_fields)
tdata.obs["is_heart"] = tdata.obs["cell_subtype"].isin(heart_fields.keys())
field_palette = {"pSHF": subtype_palette['Second heart field (posterior)'],
                 "aSHF": subtype_palette['Second heart field (anterior)'],
                 "JCF": subtype_palette['Juxtacardiac field'],
                 "Epicardium": subtype_palette['Epicardium'],
                 "Endocardial": subtype_palette['Endocardial cell'],
                 "FHF": subtype_palette['First heart field']}

## UMAPs

In [ ]:
heart_tdata = tdata[tdata.obs["is_heart"]].copy()
sc.pp.neighbors(heart_tdata, use_rep="X_scvi")
sc.tl.umap(heart_tdata)

Colored by field

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), dpi = 600)
sc.pl.umap(heart_tdata, color="cell_subtype", palette=subtype_palette,
        size=3, sort_order = False, ax = ax, title = "", frameon=False)
save_plot(plots_path / "heart_umap.svg", fig, rasterize=True)

Colored by stage

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), dpi = 600)
sc.pl.umap(heart_tdata, color="stage", palette=stage_palette,
        size=3, sort_order = False, ax = ax, title = "", frameon=False)
save_plot(plots_path / "heart_stage_umap.svg", fig, rasterize=True)

## Subtype Marker genes

In [ ]:
sc.pl.heatmap(heart_tdata, groupby= "cell_subtype", var_names =
              ["Nr2f2","Tbx5","Shox2","Tnnt2","Nkx2-5","Isl1","Rgs5","Tbx1","Hand1",
              "Mab21l2","Tbx18","Wt1","Kdr","Nfatc1","Myl3","Myh7","Myh6","Bmp2"],
              standard_scale = "var", figsize = (2.5,4), show = False)
fig = plt.gcf()
save_plot(plots_path / "heart_marker_heatmap.svg", plt.gcf(), rasterize=True, dpi = 600)

## Sankey plot

In [10]:
e95_tdata = tdata[(tdata.obs.stage == "E9.5") & tdata.obs.clone.notnull(),[]].copy()
e95_tdata.obs["is_mesoderm"] = e95_tdata.obs["germ_layer"].map(lambda x: "Mesoderm" if x == "Mesoderm" else "Other").to_dict()
e95_tdata.obs["is_heart"] = e95_tdata.obs["cell_subtype"].map(lambda x: "Heart" if x in heart_fields.keys() else "Other").to_dict()
e95_tdata.obs["filtered_heart_field"] = e95_tdata.obs["heart_field"].fillna("Other")
e95_tdata.obs.loc[e95_tdata.obs["filtered_heart_field"].isin(["Epicardium","Endocardial"]), "filtered_heart_field"] = pd.NA
e95_tdata.obs["heart_subtype"] = e95_tdata.obs["cell_subtype"].map(lambda x: x if x in heart_fields.keys() else "Other").to_dict()
germ_assignment = e95_tdata.obs["is_mesoderm"].to_dict()
heart_assignment = e95_tdata.obs["is_heart"].to_dict()
filtered_field_assignment = e95_tdata.obs["filtered_heart_field"].to_dict()
field_assignment = e95_tdata.obs["heart_field"].fillna("Other").to_dict()
subtype_assignment = e95_tdata.obs["heart_subtype"].to_dict()
for tree in e95_tdata.obst.values():
    annotate_commitment(tree, [germ_assignment, heart_assignment, field_assignment], threshold=0.9, exclude_categories=["Other"])

In [ ]:
paths = extant_transition_count(e95_tdata[e95_tdata.obs.heart_field.notnull()], key = "commitment",
                                depth_key="time", bins = np.arange(0, 10, .5))
fig, ax = plt.subplots(figsize=(2.8, 2),dpi=600)
sankey_palette = sankey_palette = {**germ_layer_palette,**field_palette,**{"Uncommitted": "#CCCCCC", "Heart": "#0015FF"}}
sankey_order = ["Uncommitted", "Mesoderm", "Heart"] + list(reversed(field_order))
plot_restriction_sankey(paths.query("source_time > 3"), normalize = True, ax = ax, order = sankey_order, palette = sankey_palette)
plt.xticks([1,5,9,13], ["4", "6", "8", "10"]);
plt.xlim(0, 13)
plt.xticks(rotation=0)
save_plot(plots_path / "e95_heart_sankey.svg", fig, rasterize=False)

## Pairwise ancestral linkage

In [ ]:
subsets = {"e8": ["E8.0"], "e85": ["E8.5"], "e9-10": ["E9.0","E9.5", "E10.0"]}
sizes = {"e8":1.2, "e85":2.3, "e9-10":3}
for name, stages in subsets.items():
    subset = heart_tdata[heart_tdata.obs["stage"].isin(stages),[]].copy()
    py.tl.ancestral_linkage(subset, groupby="cell_subtype", depth_key="time", by_tree=True, metric = "lca")
    fig, ax = plt.subplots(figsize=(sizes[name],sizes[name]), dpi=600)
    py.pl.ancestral_linkage(subset, vmax = 1.5, vmin = -1.5, cluster = "ward", ax = ax)
    save_plot(plots_path / f"{name}_heart_ancestral_linkage.svg", fig)

## Subtype ancestral linkage

In [ ]:
for subtype in heart_tdata.obs["cell_subtype"].unique():
    py.tl.ancestral_linkage(heart_tdata, groupby="cell_subtype", target = subtype, metric = "lca")
    # Set linkage for target cells
    mask = (heart_tdata.obs["cell_subtype"] == subtype) & heart_tdata.obs["clade"].notna()
    heart_tdata.obs.loc[mask, f"{subtype}_linkage"] = -(
        heart_tdata.obs.loc[mask, "clone"]
        .map(heart_tdata.uns["cell_subtype_test"].groupby("tree")["permuted_value"].mean())
        .astype(float)
    )

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12,9), dpi=600)
axes = axes.flatten()
norm = mcolors.TwoSlopeNorm(vmin=-2, vcenter=0, vmax=3)
for i, subtype in enumerate(heart_tdata.obs["cell_subtype"].unique()):
    ax = axes[i]
    sc.pl.umap(heart_tdata[heart_tdata.obs[f"{subtype}_linkage"].notnull()],
               color=f"{subtype}_linkage", s = 1.5, cmap="RdBu_r", norm=norm, sort_order=False,
        colorbar_loc = None, frameon=False, ax = ax, show = False )
save_plot(plots_path / "heart_linkage_umap.svg", fig, rasterize=True)

## Identify fate-restricted clades

Heart restricted

In [5]:
tdata.obs["is_heart"] = tdata.obs["is_heart"].astype(int)
py.tl.ancestral_states(tdata, "is_heart", method = "mean")
heart_progenitors = get_fate_progenitors(tdata, "is_heart", key_added = "heart_progenitor")
heart_progenitors["stage"] = heart_progenitors["clone"].str.split("-R").str[0]
heart_progenitors["embryo"] = heart_progenitors["clone"].str.split("-C").str[0]
tdata.obs["progenitor_field"] = tdata.obs["heart_progenitor"].map(
    tdata.obs.groupby("heart_progenitor")["heart_field"].agg(lambda x: x.mode()[0]))
heart_progenitors["field"] = heart_progenitors.node.map(
    tdata.obs.groupby("heart_progenitor")["heart_field"].agg(lambda x: x.mode()[0]))

Heart field restricted

In [6]:
tdata.obs["heart_field"] = tdata.obs["heart_field"].fillna("Other")
one_hot = pd.get_dummies(tdata.obs["heart_field"]).astype(int)
tdata.obsm["heart_field_counts"] = one_hot
tdata.obs["n"] = 1
py.tl.ancestral_states(tdata, keys = "heart_field_counts", method = "sum")
py.tl.ancestral_states(tdata, keys = "n", method = "sum")
field_progenitors = get_fate_progenitors(tdata, "heart_field_counts", key_added = "field_progenitor")
field_progenitors["stage"] = field_progenitors["clone"].str.split("-R").str[0]
field_progenitors["embryo"] = field_progenitors["clone"].str.split("-C").str[0]
field_progenitors = field_progenitors.query("fate != 'Other'").reset_index()

Germ layer restricted

In [ ]:
fate_results = base_path / "fate" / "results"
germ_layer_assignment = pd.read_csv(fate_results / "germ_layer_progenitor_descendants.csv", index_col=0)
heart_tdata.obs["germ_layer_progenitor"] = germ_layer_assignment["progenitor"]

## Plot fate-restricted clades

In [ ]:
fig, ax = plt.subplots(figsize=(1.2, 3.6), dpi = 600)
clone = 'E9.5-R2-C1'
nodes = heart_progenitors.query("clone == @clone").node.unique().tolist()
py.pl.tree(tdata, depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
py.pl.nodes(tdata,tree = clone, nodes = nodes, ax = ax, color = colors[2], size = 15, outline_width=.3)
save_plot(plots_path / "heart_progenitor_tree.svg", fig, rasterize=True)

Example clades

In [ ]:
for node in ["QfIST5JM","vdyfw5LS","KUzo0pfR"]:
    subset = heart_tdata[heart_tdata.obs.progenitor == node]
    fig, ax = plt.subplots(figsize=(1.5,subset.shape[0]/10), dpi=600)
    py.pl.tree(subset, depth_key="time", ax = ax, branch_linewidth=0.5)
    py.pl.annotation(subset, keys = "cell_subtype", palette = subtype_palette, label = False, ax = ax, gap = .05, legend = True, width = .15)
    save_plot(plots_path / f"heart_prog_tree_{node}.svg", fig)

## Clade statistics

In [12]:
prog_stats = (
    field_progenitors
    .query("stage == 'E9.5'")
    .groupby(["embryo", "stage", "fate"])
    .agg(
        mean_time=("time", "mean"),
        mean_size=("fate_descendants", "mean"),
        n_progenitors=("node", "nunique"),
    )
    .reset_index()
)
prog_stats.fate = pd.Categorical(prog_stats.fate, categories=field_order, ordered=True)

Number of clades

In [13]:
fig, ax = plt.subplots(figsize=(.728,1.912), dpi = 600)
sns.barplot(data = prog_stats, y = "fate", x = "n_progenitors", hue = "fate",legend = False, saturation = 1,edgecolor="black",
                    palette = field_palette, ax = ax,linewidth=0.5,errorbar="se",capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"})
sns.scatterplot(data = prog_stats, y = "fate", x =  "n_progenitors", color = "black", s = 6,
                    legend = False, ax=ax, edgecolor="none")
save_plot(plots_path / "heart_field_progenitor_count.svg", fig, rasterize=True)

Clade size

In [204]:
fig, ax = plt.subplots(figsize=(1.2,2), dpi = 600)
sns.barplot(data = prog_stats, y = "fate", x = "mean_size", hue = "fate",legend = False, saturation = 1,edgecolor="black",
                    palette = field_palette, ax = ax,linewidth=0.5,errorbar="se",capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"})
sns.scatterplot(data = prog_stats, y = "fate", x =  "mean_size", color = "black", s = 6,
                    legend = False, ax=ax, edgecolor="none")
save_plot(plots_path / "heart_field_progenitor_size.svg", fig, rasterize=True)

Fate restriction time

In [14]:
fig, ax = plt.subplots(figsize=(.728,1.912), dpi = 600)
sns.violinplot(data = field_progenitors.query("stage == 'E9.5'"), y = "fate", x = "time", hue = "fate",
               order = field_order, palette = field_palette, inner = "quart", linewidth=0.8, saturation = 1, ax = ax)
plt.xlabel("Inferred restriction time");
plt.ylabel("Fate");
save_plot(plots_path / "heart_field_restriction_violin.svg", fig)

## Fraction of clades specified directly from Mesoderm

In [209]:
get_commitment_path(e95_tdata,key_added = "commitment_path")
e95_tdata.obs["heart_before_field"] = e95_tdata.obs["commitment_path"].str.contains("Heart")
df = (
    e95_tdata.obs
    .query("is_heart == 'Heart'")
    .groupby("heart_field")["heart_before_field"]
    .mean()
    .reindex(reversed(field_order))   # controls heart_field order
    .reset_index()
)

df["True"] = df["heart_before_field"]
df["False"] = 1 - df["heart_before_field"]
plot_df = df.set_index("heart_field")[["True", "False"]]
fig, ax = plt.subplots(figsize=(1.2, 2), dpi=600)
plot_df.plot(
    kind="barh",
    stacked=True,
    ax=ax,
    color={"True": germ_layer_palette["Mesoderm"], "False": "lightgray"},
    width=0.8,
)

ax.set_xlim(0, 1)
ax.set_xlabel("Heart field origin")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

legend = ax.get_legend()
if legend is not None:
    legend.remove()

save_plot(plots_path / "heart_field_origin.svg", fig)

## Clade coherence

Permutation test

In [338]:
prog_coherence =  progenitor_dispersion_stratified_permutation(
    heart_tdata, n_permutations=1000, alternative="less", min_cells=5,
    strat_col = "heart_field").sort_values("observed")
prog_coherence["significant"] = prog_coherence["p_adj"] < 0.05

Histogram

In [400]:
fig, ax = plt.subplots(figsize=(1.5,1.5), dpi=600)
sns.histplot(prog_coherence, bins=20, x = "observed", hue = "significant", alpha=1,legend = False, 
    multiple="stack", hue_order = [True, False], ax=ax, palette = [colors[2], "lightgray"])
plt.xlabel("Mean pairwise dist.")
plt.ylabel("Number of prog.")
save_plot(plots_path / "heart_progenitor_coherence_histogram.svg", fig)

Example coherent clades

In [402]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
most_coherent = prog_coherence.head(20).index.values.tolist()
sc.pl.umap(heart_tdata, groups = most_coherent, ax = ax, title = "", frameon=False, show = False)
sc.pl.umap(heart_tdata[heart_tdata.obs.progenitor.isin(most_coherent)], s = 30,
    color = "progenitor", groups = most_coherent, ax = ax, title = "", frameon=False, palette = colors)
save_plot(plots_path / "heart_coherent_progenitors_umap.svg", fig, rasterize = True)

Example non-coherent clades

In [403]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
least_coherent = prog_coherence.tail(20).index.values.tolist()
sc.pl.umap(heart_tdata, groups = least_coherent, ax = ax, title = "", frameon=False, show = False)
sc.pl.umap(heart_tdata[heart_tdata.obs.progenitor.isin(least_coherent)], s = 30,
    color = "progenitor", groups = least_coherent, ax = ax, title = "", frameon=False, palette = colors)
save_plot(plots_path / "heart_noncoherent_progenitors_umap.svg", fig, rasterize = True)

## Endocardial containing clades

In [ ]:
heart_tdata.obs["contains_endocardial"] = heart_tdata.obs["heart_progenitor"].map(
    heart_tdata.obs.groupby("heart_progenitor")["heart_field"].agg(lambda x: "Endocardial" in x.values)).map({True: "True", False: "False"})
heart_tdata.obs.loc[heart_tdata.obs.cell_subtype == "Endocardial cell", "contains_endocardial"] = "True"
heart_progenitors["field_mode"] = heart_progenitors["node"].map(
    heart_tdata.obs.groupby("progenitor")["heart_field"].agg(lambda x: x.mode()[0]))
heart_tdata.obs["field_mode"] = heart_tdata.obs["progenitor"].map(heart_progenitors.set_index("node")["field_mode"])

Umap

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), dpi=600)
sc.pl.umap(heart_tdata, ax = ax, show = False)
sc.pl.umap(heart_tdata, color="contains_endocardial", groups = ["True"],
           palette = ["black"], ax = ax, title = "", frameon=False)
save_plot(plots_path / "heart_endocardial_progenitor_umap.svg", fig, rasterize=True)

Endocardial cell sources

In [ ]:
df = heart_tdata.obs.query("cell_subtype == 'Endocardial cell'").field_mode.value_counts().reset_index(name = "count")
df["count"] = df["count"] / df["count"].sum()
df["pct"] = df["count"] * 100
fig, ax = plt.subplots(figsize=(1,1), dpi=600)
sns.barplot(data = df, x = "field_mode", y = "pct", saturation=1, order = df.field_mode, palette = field_palette, ax = ax)
plt.yticks([0,20,40,60,80])
plt.ylabel("Endocardial cells (%)")
plt.xticks(rotation=90)
plt.xlabel("Heart clade bias")
save_plot(plots_path / "endocardial_field_bias.svg", fig)

## Heart clade cooccurrence

In [56]:
cooc, pmi, npmi = compute_pmi(heart_tdata.obs.query("stage != 'E8.0'"), group_col="heart_progenitor", cat_col="heart_field")
fig, ax = plt.subplots(figsize=(2, 2), dpi=600)
field_order = ["FHF", "JCF",  "aSHF","pSHF", "Endocardial","Epicardium"]
sns.heatmap(pmi.loc[field_order, field_order], square=True, xticklabels=field_order, 
            yticklabels=field_order, cmap = cooc_cmap, vmin = -2, vmax  = 2, ax = ax)
save_plot(plots_path / "heart_field_pmi_heatmap.svg", fig)

## Fate distribution heatmap

Heart-restricted clades

In [ ]:
fig = plot_progenitor_output(
    heart_tdata.obs.query("stage != 'E8.0'"),
    progenitor_col="heart_progenitor",
    fate_col="heart_field",
    fate_order=field_order,
    min_cells=2,
    row_order="cluster",
    cmap=sequential_cmap,
    vmax=0.5,
    ylim=(0, 50),
    xlabel="Heart progenitors",
)
save_plot(plots_path / "heart_prog_by_heart_field.svg",fig,rasterize=True,)

Mesoderm-restricted clades

In [ ]:
fig = plot_progenitor_output(
    heart_tdata.obs.query("stage != 'E8.0'"),
    progenitor_col="germ_layer_progenitor",
    fate_col="heart_field",
    fate_order=field_order,
    min_cells=2,
    row_order="cluster",
    cmap=sequential_cmap,
    vmax=0.5,
    ylim=(0, 100),
    xlabel="Mesodermal progenitors",
)

save_plot(plots_path / "germ_layer_prog_by_heart_field.svg",fig,rasterize=True)